# Etapa 1 — modelo de churn, inflexão e seleção dos grupos

Objetivo: usar dados históricos para construir um score fora da amostra, estimar um cutoff operacional e selecionar uma população local para um piloto randomizado.

**Atenção:** proximidade do cutoff não identifica efeito causal. Ela define quem é elegível; a randomização 1:1 é o que torna controle e tratamento comparáveis. Em produção, treine na história e aplique o modelo congelado a uma coorte futura sem outcome observado.

## 1. Configuração

A fração de candidatos controla a largura da janela. Comece com capacidade operacional e cálculo de poder, não escolhendo a faixa pelo resultado causal.

In [ ]:
from pathlib import Path

DATA_PATH = Path("data/churn.csv")
ID_COLUMN = "customer_id"
OUTCOME_COLUMN = "churn_90d"
FEATURE_COLUMNS = ["tenure", "complaints", "usage_change"]
CATEGORICAL_COLUMNS = []
N_FOLDS = 5
CANDIDATE_FRACTION = 0.30
RANDOM_STATE = 42
OUTPUT_PATH = Path("data/churn_pilot_assignment.csv")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.base import clone
from sklearn.calibration import calibration_curve
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")


## 2. Base histórica e validações

`acao` e variáveis posteriores à ação ficam fora das features. Se o churn histórico já foi afetado por campanhas, o modelo prevê o risco sob a política histórica — não necessariamente o risco sem intervenção.

In [ ]:
def generate_synthetic_churn(path: Path, n: int = 1_500, seed: int = 42) -> pd.DataFrame:
    """Gera uma base didática; nunca deve ser interpretada como evidência real."""
    rng = np.random.default_rng(seed)
    tenure = np.clip(np.rint(rng.gamma(2.0, 10.0, n)), 1, 72).astype(int)
    complaints = np.clip(rng.poisson(0.7, n), 0, 4)
    usage_change = np.clip(rng.normal(-0.05, 0.23, n), -0.8, 0.7)
    logit = -0.75 - 0.025 * tenure + 0.52 * complaints - 2.1 * usage_change
    churn_probability = 1 / (1 + np.exp(-logit))
    generated = pd.DataFrame({
        ID_COLUMN: np.arange(1, n + 1),
        OUTCOME_COLUMN: rng.binomial(1, churn_probability),
        "tenure": tenure,
        "complaints": complaints,
        "usage_change": usage_change.round(4),
    })
    path.parent.mkdir(parents=True, exist_ok=True)
    generated.to_csv(path, index=False)
    return generated

if not DATA_PATH.exists():
    df = generate_synthetic_churn(DATA_PATH, seed=RANDOM_STATE)
    print(f"Base sintética criada em {DATA_PATH.resolve()}")
else:
    df = pd.read_csv(DATA_PATH)

required = {ID_COLUMN, OUTCOME_COLUMN, *FEATURE_COLUMNS}
missing = required.difference(df.columns)
if missing:
    raise ValueError(f"Colunas ausentes: {sorted(missing)}")
if df[ID_COLUMN].duplicated().any():
    raise ValueError("A unidade de randomização precisa ter um ID único")
if not set(df[OUTCOME_COLUMN].dropna().unique()).issubset({0, 1}):
    raise ValueError(f"{OUTCOME_COLUMN} deve ser 0/1")

development = df.dropna(subset=[OUTCOME_COLUMN, *FEATURE_COLUMNS]).copy()
X = pd.get_dummies(
    development[FEATURE_COLUMNS], columns=CATEGORICAL_COLUMNS,
    drop_first=False, dtype=float
)
y = development[OUTCOME_COLUMN].astype(int).to_numpy()
print(f"N histórico: {len(development):,}; churn: {y.mean():.2%}")
display(development[FEATURE_COLUMNS + [OUTCOME_COLUMN]].describe().T)


## 3. Modelo-base com cross-fitting

Cada cliente recebe uma probabilidade de um modelo que não foi treinado com seu próprio outcome. Isso evita a avaliação otimista in-sample. Para dados temporais, substitua `StratifiedKFold` por divisões temporais que nunca treinem no futuro.

In [ ]:
base_model = HistGradientBoostingClassifier(
    max_iter=250, learning_rate=0.04, max_leaf_nodes=15,
    min_samples_leaf=30, l2_regularization=1.0,
    random_state=RANDOM_STATE,
)
cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
oof_score = cross_val_predict(
    base_model, X, y, cv=cv, method="predict_proba", n_jobs=-1
)[:, 1]
development["score_oof"] = oof_score

metrics = pd.Series({
    "roc_auc_oof": roc_auc_score(y, oof_score),
    "brier_oof": brier_score_loss(y, oof_score),
    "score_min": oof_score.min(),
    "score_max": oof_score.max(),
})
display(metrics.round(4).to_frame("valor"))


## 4. Cutoff pela inflexão da calibração

Ajustamos `P(churn=1 | logit(score))` com regressão logística. A sigmoide tem sua inflexão quando o risco calibrado é 50%; resolvemos qual score bruto corresponde a esse ponto. Se 50% estiver fora do suporte dos scores, o cutoff é inadequado e o notebook interrompe.

Isso é uma definição operacional. Um modelo de ML genérico não possui necessariamente um único “ponto de inflexão” intrínseco.

In [ ]:
eps = 1e-6
logit_score = np.log(np.clip(oof_score, eps, 1-eps) / np.clip(1-oof_score, eps, 1-eps)).reshape(-1, 1)
calibrator = LogisticRegression(C=1e6, max_iter=2_000).fit(logit_score, y)
slope = float(calibrator.coef_[0, 0])
intercept = float(calibrator.intercept_[0])
if slope <= 0:
    raise ValueError("Curva de calibração não é crescente; revise o modelo-base")
cutoff_logit = -intercept / slope
inflection_cutoff = float(1 / (1 + np.exp(-cutoff_logit)))
if not (oof_score.min() <= inflection_cutoff <= oof_score.max()):
    raise ValueError(
        f"Inflexão {inflection_cutoff:.3f} fora do suporte observado; "
        "use um cutoff operacional ou por capacidade"
    )

prob_true, prob_pred = calibration_curve(y, oof_score, n_bins=10, strategy="quantile")
grid = np.linspace(oof_score.min(), oof_score.max(), 300)
grid_logit = np.log(np.clip(grid, eps, 1-eps) / np.clip(1-grid, eps, 1-eps)).reshape(-1, 1)
calibrated_grid = calibrator.predict_proba(grid_logit)[:, 1]

plt.figure(figsize=(9, 5))
plt.plot(prob_pred, prob_true, "o", label="calibração OOF por decis")
plt.plot(grid, calibrated_grid, label="curva logística de calibração")
plt.axvline(inflection_cutoff, color="crimson", linestyle="--", label=f"inflexão = {inflection_cutoff:.3f}")
plt.axhline(0.5, color="grey", linestyle=":", linewidth=1)
plt.xlabel("Score OOF bruto")
plt.ylabel("Probabilidade calibrada / churn observado")
plt.legend()
plt.show()
print(f"Cutoff de inflexão no score bruto: {inflection_cutoff:.4f}")


## 5. Janela local e alocação 1:1

Selecionamos a fração configurada de clientes com menor distância absoluta ao cutoff. Dentro de estratos do score, embaralhamos IDs e alternamos os braços. O outcome não participa dessa alocação.

Para o piloto real, execute essa lógica em uma nova coorte usando scores do modelo final congelado. A base histórica abaixo demonstra o mecanismo, mas não transforma intervenções passadas em um experimento.

In [ ]:
if not 0 < CANDIDATE_FRACTION <= 1:
    raise ValueError("CANDIDATE_FRACTION deve estar em (0, 1]")

development["distance_to_cutoff"] = (development["score_oof"] - inflection_cutoff).abs()
n_candidates = max(2 * N_FOLDS, int(round(len(development) * CANDIDATE_FRACTION)))
candidates = development.nsmallest(n_candidates, "distance_to_cutoff").copy()
bandwidth = float(candidates["distance_to_cutoff"].max())

# Estratificação preserva aproximadamente a distribuição do risco nos dois braços.
n_strata = min(10, max(2, len(candidates) // 20))
candidates["score_stratum"] = pd.qcut(
    candidates["score_oof"], q=n_strata, labels=False, duplicates="drop"
)
rng = np.random.default_rng(RANDOM_STATE)
candidates["random_order"] = rng.random(len(candidates))
candidates = candidates.sort_values(["score_stratum", "random_order", ID_COLUMN])
candidates["assigned_group"] = (
    (candidates.groupby("score_stratum", observed=True).cumcount()
     + candidates["score_stratum"].astype(int)).mod(2)
    .map({0: "controle", 1: "tratamento"})
)

print(f"Cutoff: {inflection_cutoff:.4f}; banda: ±{bandwidth:.4f}")
display(candidates.groupby("assigned_group").agg(
    n=(ID_COLUMN, "size"), score_medio=("score_oof", "mean"),
    score_min=("score_oof", "min"), score_max=("score_oof", "max")
).round(4))


## 6. Checagem de balanceamento e exportação

Diferenças padronizadas próximas de zero são esperadas pela randomização, mas não devem ser usadas para refazer repetidamente o sorteio. Registre a semente e preserve a atribuição.

In [ ]:
def standardized_mean_difference(data, column):
    control = data.loc[data["assigned_group"].eq("controle"), column].astype(float)
    treated = data.loc[data["assigned_group"].eq("tratamento"), column].astype(float)
    pooled_sd = np.sqrt((control.var(ddof=1) + treated.var(ddof=1)) / 2)
    return 0.0 if pooled_sd == 0 else (treated.mean() - control.mean()) / pooled_sd

numeric_balance = [c for c in FEATURE_COLUMNS if c not in CATEGORICAL_COLUMNS]
balance = pd.DataFrame({
    "feature": [*numeric_balance, "score_oof"],
    "smd_treatment_minus_control": [
        standardized_mean_difference(candidates, c)
        for c in [*numeric_balance, "score_oof"]
    ],
})
display(balance.round(4))

export_columns = [
    ID_COLUMN, "assigned_group", "score_oof", "score_stratum",
    "distance_to_cutoff", *FEATURE_COLUMNS,
]
assignment = candidates[export_columns].sort_values(ID_COLUMN).reset_index(drop=True)
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
assignment.to_csv(OUTPUT_PATH, index=False)
print(f"Alocação salva em {OUTPUT_PATH}: {len(assignment):,} participantes")
display(assignment.head())


## Próximo passo

Antes de disparar ações, registre protocolo, tratamento concreto, custo, horizonte do churn, exclusões, poder estatístico e possíveis clusters de interferência. Após o acompanhamento, estime primeiro o ITT pela atribuição randomizada. DML pode melhorar precisão ou tratar desvios, mas não substitui o sorteio.